# NLP in Python with NLTK: Code-Only Notebook

**Notebook 2 of 2.** The complete working code for all four parts, NLTK only, with comments only. Run top to bottom. The first cell installs and imports everything. Cells marked `EXTRA` in their first comment are the optional *Going deeper* examples.

# Part 1: Foundations, Regular Expressions and Tokenisation

In [ ]:
# 1.0 SETUP: installs go into the notebook's own environment (works in Colab and VS Code)
%pip install -q nltk scikit-learn pandas matplotlib numpy

# Standard library imports
import re
import math
import time
import random
import pickle
import string
import textwrap
import unicodedata
import warnings
from collections import Counter, defaultdict

# Data and plotting (these do not process language, they hold and draw results)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# NLTK: download the data packages used in this course (quiet = no noisy output).
# Older package names are included too, so this also works on older NLTK versions.
import nltk
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4",
            "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng",
            "maxent_ne_chunker", "maxent_ne_chunker_tab", "words",
            "gutenberg", "movie_reviews", "reuters", "vader_lexicon"]:
    nltk.download(pkg, quiet=True)

# The NLTK names we will use again and again
from nltk import FreqDist, ngrams, word_tokenize, sent_tokenize, pos_tag, ne_chunk
from nltk.corpus import stopwords, gutenberg, movie_reviews, reuters, wordnet

warnings.filterwarnings("ignore")
random.seed(42)
np.random.seed(42)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 140)

print("Setup complete. NLTK version:", nltk.__version__)
print(word_tokenize("NLTK is ready, and so are we!"))

In [ ]:
# 1.1 WHAT IS NLP: plain Python string operations on a tiny text
text = "Natural Language Processing helps computers understand text. NLP is everywhere!"

print("Characters:", len(text))
print("Lowercase :", text.lower())
print("Split     :", text.split())          # naive tokenisation on whitespace
print("Words     :", len(text.split()))

# Notice the problems: 'text.' and 'everywhere!' keep their punctuation,
# and 'NLP' and 'nlp' would be counted as different words if we skipped lower().
print(Counter(text.lower().split()).most_common(3))

In [ ]:
# 1.1 EXTRA: THE PIPELINE ON ONE EMAIL, STAGE BY STAGE
email = "My order #4521 arrived late and the box was damaged!!"
print("1. Raw       :", email)

# 2. Clean and split: NLTK tokeniser, lowercase, keep only words
tokens = [t.lower() for t in word_tokenize(email) if t.isalpha()]
print("2. Tokens    :", tokens)

# 3. Remove very common words using NLTK's English stopword list
stops = set(stopwords.words("english"))
content = [t for t in tokens if t not in stops]
print("3. Content   :", content)

# 4. Turn into numbers: a count for every word in a small vocabulary
vocab = ["order", "arrived", "late", "box", "damaged", "great", "happy"]
counts = FreqDist(content)
print("4. Numbers   :", {w: counts[w] for w in vocab})

# 5. A crude rule-based "model": flag a complaint if any complaint word appears
complaint_words = {"late", "damaged", "broken", "refund"}
score = sum(1 for t in content if t in complaint_words)
print("5. Complaint?:", "yes" if score >= 1 else "no", f"(score {score})")

# The rule breaks easily. Here it wrongly flags a happy message:
happy = [t.lower() for t in word_tokenize("the box was not damaged and arrived early, lovely service") if t.isalpha()]
print("Rule on a happy message:", sum(1 for t in happy if t in complaint_words), "(wrongly flagged)")
# A model learns weights from labelled examples instead of us writing rules by hand.

In [ ]:
# 1.2 REGEX BASICS

# search vs match vs fullmatch
s = "NLP in 2026"
print(re.search(r"\d+", s))        # finds '2026' anywhere
print(re.match(r"\d+", s))         # None, because the string does not START with a digit
print(re.fullmatch(r"\w+ \w+ \d+", s) is not None)

# The wildcard and character classes
print(re.findall(r"c.t", "cat cot cut c t cart"))      # '.' is any single character
print(re.findall(r"c[aeiou]t", "cat cot cut cyt"))      # only vowels in the middle
print(re.findall(r"[^aeiou\s]+", "regular expressions")) # runs of non-vowels

# Quantifiers
print(re.findall(r"\d+", "Room 12, floor 3, ext 4567"))          # one or more digits
print(re.findall(r"\d{4}", "Room 12, floor 3, ext 4567"))        # exactly four digits
print(re.findall(r"colou?r", "color colour colouur"))            # 'u' optional

# Anchors
print(bool(re.search(r"^Dear", "Dear Hari, hello")))
print(bool(re.search(r"hello$", "Dear Hari, hello")))

# Groups: pull out parts of a match
m = re.search(r"(\d{4})-(\d{2})-(\d{2})", "Course starts 2026-11-05.")
print(m.group(0), "| year:", m.group(1), "| month:", m.group(2), "| day:", m.group(3))

In [ ]:
# 1.2 EXTRA: MORE REGEX TOOLS

# Word boundary \b: match 'cat' as a whole word only
s = "cat concatenate scatter cat."
print(re.findall(r"cat", s))        # finds 'cat' inside other words too
print(re.findall(r"\bcat\b", s))    # whole word only

# Alternation | means 'or' (note the capital C in 'Colour' is not matched)
print(re.findall(r"colou?r|grey|gray", "Colour grey gray color"))

# Case-insensitive search
print(re.findall(r"colou?r", "Colour color COLOUR", flags=re.IGNORECASE))

# Greedy vs lazy: .* grabs as much as possible, .*? as little as possible
html = "<b>bold</b> and <i>italic</i>"
print(re.findall(r"<.*>", html))     # greedy: one big match
print(re.findall(r"<.*?>", html))    # lazy: each tag separately

# Special characters need a backslash when you mean them literally
print(re.findall(r"\d+\.\d+", "Pi is 3.14, area 3x14"))   # \. is a real full stop
print(re.findall(r"\d+.\d+", "Pi is 3.14, area 3x14"))    # plain . matches ANY character

# finditer also tells you where each match is
for m in re.finditer(r"\d+", "Room 12, floor 3"):
    print(m.group(), "starts at", m.start(), "ends at", m.end())

# Split text on a pattern
print(re.split(r"[,;]\s*", "apples, pears; plums,grapes"))

In [ ]:
# 1.3 REGEX IN PRACTICE
post = ("Loving the new #NLP course! Email hari@example.com or message @skilledotter "
        "before 2026-11-05. Early bird price: £149.99 #Python #AI https://example.com/course")

# Hashtags: a '#' followed by word characters
hashtags = re.findall(r"#\w+", post)
print("Hashtags :", hashtags)

# Mentions, first attempt (has a bug)
print("Mentions v1:", re.findall(r"@\w+", post))      # also grabs '@example' from the email

# Fix: only match '@' when it is NOT directly preceded by a non-space character
print("Mentions v2:", re.findall(r"(?<!\S)@\w+", post))

# Emails
print("Emails   :", re.findall(r"[\w.+-]+@[\w-]+\.[\w.]+", post))

# ISO dates and prices
print("Dates    :", re.findall(r"\d{4}-\d{2}-\d{2}", post))
print("Prices   :", re.findall(r"£\d+(?:\.\d{2})?", post))

# URLs
urls = re.findall(r"https?://\S+", post)
print("URLs     :", urls)

# Cleaning with re.sub: remove URLs, emails, then the symbols, then extra spaces
clean = re.sub(r"https?://\S+", "", post)
clean = re.sub(r"[\w.+-]+@[\w-]+\.[\w.]+", "", clean)
clean = re.sub(r"[#@£]", "", clean)
clean = re.sub(r"\s+", " ", clean).strip()
print("\nCleaned  :", clean)

# Compile a pattern once if you will use it many times
hashtag_re = re.compile(r"#\w+")
print(hashtag_re.findall("#one and #two"))

In [ ]:
# 1.3 EXTRA: NAMED GROUPS INTO A TABLE, WITH A COMMENTED PATTERN
log = """2026-10-26 08:15 ERROR payment failed for order 4521 (user hari@example.com)
2026-10-26 08:17 INFO order 4522 shipped
2026-10-26 08:19 ERROR stock check failed for order 4523 (user sam@example.org)"""

# (?P<name>...) names a group. re.VERBOSE ignores layout whitespace and allows # comments
line_pattern = re.compile(r"""
    (?P<date>\d{4}-\d{2}-\d{2})\s+          # date
    (?P<time>\d{2}:\d{2})\s+                # time
    (?P<level>[A-Z]+)\s+                    # level such as ERROR or INFO
    (?:(?P<message>.*?)\s+)?                # optional message text, as little as needed
    order\s+(?P<order>\d+)                  # order number
    """, re.VERBOSE)

rows = [m.groupdict() for m in line_pattern.finditer(log)]
log_df = pd.DataFrame(rows)
print(log_df)

# Count errors per level, as you would on a real log file
print(log_df["level"].value_counts())

In [ ]:
# 1.4 TOKENISATION WITH NLTK
from nltk.tokenize import (word_tokenize, sent_tokenize, TweetTokenizer, RegexpTokenizer,
                           WhitespaceTokenizer, WordPunctTokenizer, MWETokenizer)

text = "Dr. Smith didn't visit London on 5 Nov. He's coming back next week! Isn't that great?"

# 1. Plain Python
print("split()        :", text.split())

# 2. Regex: runs of word characters only (loses apostrophes and punctuation)
print("regex \\w+      :", re.findall(r"\w+", text))
# A better regex: keep apostrophes inside words
print("regex better   :", re.findall(r"\w+(?:'\w+)?", text))

# 3. The NLTK family
print("Whitespace     :", WhitespaceTokenizer().tokenize(text))     # same as split()
print("WordPunct      :", WordPunctTokenizer().tokenize(text))      # splits "didn't" into didn ' t
print("RegexpTokenizer:", RegexpTokenizer(r"\w+(?:'\w+)?").tokenize(text))   # your own pattern
print("word_tokenize  :", word_tokenize(text))                      # splits "didn't" into did + n't
print("sent_tokenize  :", sent_tokenize(text))                      # knows 'Dr.' does not end a sentence

# NLTK tokeniser built for tweets: keeps hashtags, mentions and emoticons as single tokens
tweet = "Loving the #NLP course @skilledotter :) so much!!!"
print("\nTweet, default :", word_tokenize(tweet))
print("TweetTokenizer :", TweetTokenizer().tokenize(tweet))

# Multi-word expressions: glue chosen word pairs back together after tokenising
mwe = MWETokenizer([("New", "York"), ("machine", "learning")], separator="_")
print("\nMWETokenizer   :", mwe.tokenize(word_tokenize("I study machine learning in New York")))

# A small table of what each token looks like (we meet part of speech tagging properly in Part 2)
tokens = word_tokenize(text)
pd.DataFrame([(t, t.isalpha(), t in string.punctuation, tag) for t, tag in pos_tag(tokens)],
             columns=["token", "is_alpha", "is_punct", "part_of_speech"]).head(10)

In [ ]:
# 1.4 EXTRA: ONE AWKWARD TEXT, SIX TOKENISERS
tricky = "State-of-the-art NLP costs £1,000.50 :) Email me@example.com, it's 5pm in New York!"

results = {
    "split()":         tricky.split(),
    "regex \\w+":      re.findall(r"\w+", tricky),
    "WordPunct":       WordPunctTokenizer().tokenize(tricky),
    "word_tokenize":   word_tokenize(tricky),
    "TweetTokenizer":  TweetTokenizer().tokenize(tricky),
    "RegexpTokenizer": RegexpTokenizer(r"[\w'£.,-]+|[:)]+|[^\w\s]").tokenize(tricky),
}
for name, toks in results.items():
    print(f"{name:<16} {len(toks):>2} tokens: {toks}")
# Same sentence, different counts. Decide which one suits your task.

In [ ]:
# 1.5 EXPLORING A TEXT WITH FREQDIST
from nltk.text import Text

raw = gutenberg.raw("carroll-alice.txt")
print("Characters:", len(raw))
print(raw[:300].replace("\n", " "), "...")

# Tokenise and keep only alphabetic tokens, lowercased
tokens = [t.lower() for t in word_tokenize(raw) if t.isalpha()]
fdist = FreqDist(tokens)                                   # NLTK's frequency counter
print("\nTotal tokens :", fdist.N())
print("Vocabulary   :", fdist.B())                          # B = number of distinct bins (types)
print("Lexical diversity:", round(fdist.B() / fdist.N(), 3))
print("\nTop 10:", fdist.most_common(10))
print("Relative frequency of 'alice':", round(fdist.freq("alice"), 4))

# Chart 1: distribution of word lengths
word_lengths = [len(t) for t in tokens]
plt.figure(figsize=(7, 3.5))
plt.hist(word_lengths, bins=range(1, 16), edgecolor="black")
plt.title("Word length distribution (Alice in Wonderland)")
plt.xlabel("Characters in word")
plt.ylabel("Number of tokens")
plt.show()

# Chart 2: distribution of sentence length (in words)
sentences = sent_tokenize(raw)
sent_lengths = [len(word_tokenize(s)) for s in sentences]
plt.figure(figsize=(7, 3.5))
plt.hist(sent_lengths, bins=30, edgecolor="black")
plt.title("Sentence length distribution")
plt.xlabel("Tokens in sentence")
plt.ylabel("Number of sentences")
plt.show()

# Chart 3: top 15 words as a bar chart
words, counts = zip(*fdist.most_common(15))
plt.figure(figsize=(8, 3.5))
plt.bar(words, counts)
plt.xticks(rotation=45)
plt.title("Top 15 words, nothing removed yet")
plt.show()

# NLTK's Text object adds reading tools. Concordance shows a word in context
Text(gutenberg.words("carroll-alice.txt")).concordance("rabbit", lines=4)

In [ ]:
# 1.5 EXTRA: ZIPF'S LAW. A few words are very common, most are rare.
ranked = fdist.most_common()
ranks = np.arange(1, len(ranked) + 1)
counts_sorted = np.array([c for _, c in ranked])

plt.figure(figsize=(6.5, 4))
plt.loglog(ranks, counts_sorted)
plt.xlabel("Rank of word (1 = most common)")
plt.ylabel("Frequency")
plt.title("Rank vs frequency on log scales")
plt.show()

# Words that appear exactly once ("hapax legomena"): NLTK has a method for it
hapax = fdist.hapaxes()
print("Words appearing once:", len(hapax), "of", fdist.B(),
      f"({len(hapax) / fdist.B():.0%} of the vocabulary)")
print("Examples:", hapax[:10])
print("The top 10 words cover", f"{counts_sorted[:10].sum() / counts_sorted.sum():.0%}", "of all tokens")

# Part 2: Text Preprocessing and Turning Text into Features

In [ ]:
# 2.1 NORMALISATION AND STOPWORDS WITH NLTK
text = "The Quick brown fox, 3 times, jumped OVER the lazy dog... and wasn't tired!"

# Step 1: lowercase and tokenise
tokens = word_tokenize(text.lower())
print("Tokens   :", tokens)

# Step 2: keep alphabetic tokens only (drops punctuation and numbers)
alpha_tokens = [t for t in tokens if t.isalpha()]
print("Alphabetic:", alpha_tokens)

# Step 3: remove stopwords using NLTK's English list
nltk_stops = set(stopwords.words("english"))
print("\nNLTK list size :", len(nltk_stops))
print("Sample         :", sorted(nltk_stops)[:12])
filtered = [t for t in alpha_tokens if t not in nltk_stops]
print("After removal  :", filtered)

# NLTK has stopword lists for many languages
print("\nLanguages available:", len(stopwords.fileids()), "e.g.", stopwords.fileids()[:8])
print("Sizes:", {lang: len(stopwords.words(lang)) for lang in ["english", "french", "german", "spanish"]})

# Does removing stopwords change the top words in Alice?
alice = [w.lower() for w in gutenberg.words("carroll-alice.txt") if w.isalpha()]
print("\nTop 8 raw     :", FreqDist(alice).most_common(8))
print("Top 8 filtered:", FreqDist(w for w in alice if w not in nltk_stops).most_common(8))

# Pitfall: negation words are stopwords
print("\n'not' is a stopword:", "not" in nltk_stops)

In [ ]:
# 2.1 EXTRA: UNICODE ACCENTS AND KEEPING NEGATIONS
def strip_accents(s):
    """Turn 'café' into 'cafe' (helps when users type with and without accents)."""
    decomposed = unicodedata.normalize("NFKD", s)
    return "".join(ch for ch in decomposed if not unicodedata.combining(ch))

print(strip_accents("Café déjà vu, naïve résumé, Zürich"))

# Custom stopword list: remove the negations from the list so they are kept
negations = {"no", "nor", "not", "never"}
my_stops = nltk_stops - negations
review = "the plot was not good and the acting was never convincing"
toks = word_tokenize(review)
print("Default list   :", [t for t in toks if t not in nltk_stops])   # 'not' has disappeared
print("Keep negations :", [t for t in toks if t not in my_stops])

In [ ]:
# 2.2 STEMMING AND LEMMATISATION WITH NLTK
from nltk.stem import PorterStemmer, SnowballStemmer, LancasterStemmer, WordNetLemmatizer

words = ["running", "runs", "ran", "studies", "studying", "better", "geese", "happily", "was"]

porter = PorterStemmer()
snowball = SnowballStemmer("english")
lancaster = LancasterStemmer()          # the most aggressive of the three
wnl = WordNetLemmatizer()

rows = []
for w in words:
    rows.append({
        "word": w,
        "porter": porter.stem(w),
        "snowball": snowball.stem(w),
        "lancaster": lancaster.stem(w),
        "wordnet (noun)": wnl.lemmatize(w),              # default part of speech is noun
        "wordnet (verb)": wnl.lemmatize(w, pos="v"),     # tell it it is a verb
    })
pd.DataFrame(rows)

In [ ]:
# 2.2 EXTRA: OVER-STEMMING AND UNDER-STEMMING
related = ["university", "universe", "universal", "universities"]
print("Porter stems    :", [porter.stem(w) for w in related])      # unrelated words squashed together
print("Lancaster stems :", [lancaster.stem(w) for w in related])

variants = ["running", "ran", "runs", "run"]
print("\nPorter stems  :", [porter.stem(w) for w in variants])     # 'ran' is left alone (under-stemming)
print("Verb lemmas   :", [wnl.lemmatize(w, pos="v") for w in variants])   # all become 'run'

In [ ]:
# 2.2b PART OF SPEECH TAGGING AND POS-AWARE LEMMATISING
def wn_pos(tag):
    """Convert a Penn Treebank tag (from pos_tag) into the tag WordNet expects."""
    if tag.startswith("J"):
        return wordnet.ADJ
    if tag.startswith("V"):
        return wordnet.VERB
    if tag.startswith("R"):
        return wordnet.ADV
    return wordnet.NOUN               # default, including real nouns

sentence = "The geese were running faster than the studies suggested, and she was happier."
tagged = pos_tag(word_tokenize(sentence))           # [(word, penn_tag), ...]

pd.DataFrame([(w, tag, wnl.lemmatize(w.lower()), wnl.lemmatize(w.lower(), wn_pos(tag)))
              for w, tag in tagged if w.isalpha()],
             columns=["token", "penn_tag", "lemma (no POS)", "lemma (with POS)"])

In [ ]:
# 2.2b EXTRA: THE SAME SPELLING, DIFFERENT ROLE, DIFFERENT LEMMA
for sentence in ["I am meeting a friend.", "The meeting was long.",
                 "The leaves fell slowly.", "He leaves tomorrow."]:
    tagged = pos_tag(word_tokenize(sentence))
    print([(w, tag, wnl.lemmatize(w.lower(), wn_pos(tag))) for w, tag in tagged if w.isalpha()])
# Look at 'meeting' and 'leaves' in each sentence: tag and lemma follow the role.
# The tagger is statistical, so it is occasionally wrong. Check a sample by eye.

In [ ]:
# 2.3 PREPROCESSING FUNCTIONS
from nltk.tokenize import RegexpTokenizer

stop_set = set(stopwords.words("english"))
quick_tokenizer = RegexpTokenizer(r"[a-z]+")           # lowercase letters only

def preprocess_quick(text):
    """Fastest: regex tokens, lowercase, drop stopwords."""
    return [t for t in quick_tokenizer.tokenize(text.lower()) if t not in stop_set]

def preprocess_stem(text):
    """word_tokenize, keep letters, drop stopwords, Porter stem."""
    return [porter.stem(t) for t in word_tokenize(text.lower()) if t.isalpha() and t not in stop_set]

def preprocess(text):
    """Cleanest: tag first (capitals help the tagger), then lemmatise with the part of speech."""
    tagged = pos_tag(word_tokenize(text))
    return [wnl.lemmatize(w.lower(), wn_pos(tag)) for w, tag in tagged
            if w.isalpha() and w.lower() not in stop_set]

samples = [
    "The cats were sitting on the mats, watching birds outside.",
    "Customers are complaining that the delivery arrived late again!",
    "She has been studying machine learning for three years.",
]
print("quick:", [preprocess_quick(s) for s in samples])
print("stem :", [preprocess_stem(s) for s in samples])
print("lemma:", [preprocess(s) for s in samples])

# Speed comparison on a bigger collection (300 sentences from Alice)
big = sent_tokenize(raw)[:300]
for name, fn in [("quick", preprocess_quick), ("stem", preprocess_stem), ("lemma + POS", preprocess)]:
    t0 = time.time()
    _ = [fn(s) for s in big]
    print(f"{name:<12} {time.time() - t0:.2f}s for 300 sentences")

In [ ]:
# 2.3 EXTRA: BEFORE AND AFTER, SIDE BY SIDE
pd.DataFrame({
    "original": samples,
    "quick": [" ".join(preprocess_quick(s)) for s in samples],
    "stem": [" ".join(preprocess_stem(s)) for s in samples],
    "lemma + POS": [" ".join(preprocess(s)) for s in samples],
})

In [ ]:
# 2.4 BAG OF WORDS
docs = [
    "The product arrived quickly and works well",
    "Terrible product, arrived late and broken",
    "Quick delivery, great product, works great",
    "Late delivery and poor service",
]
doc_tokens = [preprocess(d) for d in docs]            # our Part 2 function: clean lemmas

# By hand with Counter: one dictionary per document
for toks in doc_tokens[:2]:
    print(Counter(toks))

# With NLTK's FreqDist (a Counter built for text)
print(FreqDist(doc_tokens[2]).most_common(3))

def build_dtm(token_docs, min_df=1, max_df=1.0, binary=False):
    """Build a document-term matrix (a pandas table) from lists of tokens."""
    n_docs = len(token_docs)
    doc_freq = FreqDist()
    for toks in token_docs:
        doc_freq.update(set(toks))                     # each word counted once per document
    vocab = sorted(w for w, df in doc_freq.items() if df >= min_df and df / n_docs <= max_df)
    rows = []
    for toks in token_docs:
        counts = FreqDist(toks)
        rows.append([(1 if counts[w] else 0) if binary else counts[w] for w in vocab])
    return pd.DataFrame(rows, columns=vocab, index=[f"doc{i}" for i in range(n_docs)])

dtm = build_dtm(doc_tokens)
print("\nMatrix shape:", dtm.shape, "(documents x vocabulary)")
dtm

In [ ]:
# 2.4 EXTRA: SPARSITY AND VOCABULARY CONTROL
filled = (dtm.values > 0).mean()
print(f"Toy matrix: {dtm.shape[0]} docs x {dtm.shape[1]} terms, {filled:.0%} of cells are non-zero")

# On a bigger collection the matrix is mostly zeros
alice_sents = [preprocess_quick(s) for s in sent_tokenize(raw)[:1500]]
big_dtm = build_dtm(alice_sents)
print(f"{len(alice_sents)} Alice sentences:", big_dtm.shape,
      f"-> only {(big_dtm.values > 0).mean():.2%} of cells are non-zero")

# min_df and max_df shrink the vocabulary
for kwargs in [dict(), dict(min_df=2), dict(min_df=5), dict(min_df=5, max_df=0.05)]:
    size = build_dtm(alice_sents, **kwargs).shape[1]
    print(f"{str(kwargs):<32} vocabulary = {size}")

# binary=True records presence (1) instead of the count
print(build_dtm([["buy", "buy", "buy", "now"]], binary=True))

In [ ]:
# 2.5 N-GRAMS
sentence = "this movie was not good at all"
words = sentence.split()
print("Unigrams:", list(ngrams(words, 1)))          # N = 1, single words
print("Bigrams :", list(ngrams(words, 2)))
print("Trigrams:", list(ngrams(words, 3)))

# everygrams gives all sizes at once
from nltk import everygrams
print("\nAll 1 to 2 grams:", [" ".join(g) for g in everygrams(words, 1, 2)])

# Most common bigrams in Alice (stopwords removed)
alice_clean = [w for w in alice if w not in stop_set]
print("\nTop bigrams in Alice:", FreqDist(ngrams(alice_clean, 2)).most_common(8))

# How much bigger does the vocabulary get? Count distinct n-grams of each size
alice_sentences = [quick_tokenizer.tokenize(s.lower()) for s in sent_tokenize(raw)[:1500]]
rows = []
for n in (1, 2, 3):
    grams = FreqDist(g for toks in alice_sentences for g in ngrams(toks, n))
    rows.append((n, grams.B(), f"{len(grams.hapaxes()) / grams.B():.0%}"))
pd.DataFrame(rows, columns=["n", "distinct n-grams", "appear only once"])

In [ ]:
# 2.5 EXTRA: COLLOCATIONS AND CHARACTER N-GRAMS
from nltk.collocations import BigramCollocationFinder
from nltk.metrics import BigramAssocMeasures

finder = BigramCollocationFinder.from_words(alice)
finder.apply_freq_filter(5)                                   # ignore pairs seen fewer than 5 times
print("By raw frequency:", [" ".join(b) for b in finder.nbest(BigramAssocMeasures.raw_freq, 6)])
print("By PMI           :", [" ".join(b) for b in finder.nbest(BigramAssocMeasures.pmi, 6)])
# Raw frequency gives 'of the', 'said the'. PMI finds real expressions like 'golden key', 'kid gloves'.

# Character n-grams: slice each word into 3-letter pieces
def char_grams(word, n=3):
    padded = f" {word} "
    return {padded[i:i + n] for i in range(len(padded) - n + 1)}

def similarity(a, b):
    ga, gb = char_grams(a), char_grams(b)
    return len(ga & gb) / len(ga | gb)

print("\ncolour vs color :", round(similarity("colour", "color"), 2))
print("colour vs colur :", round(similarity("colour", "colur"), 2))      # a typo still shares pieces
print("colour vs table :", round(similarity("colour", "table"), 2))

In [ ]:
# 2.6 FEATURE ENGINEERING AND LANGUAGE DETECTION WITH NLTK STOPWORD LISTS
LANGUAGES = ["english", "french", "german", "spanish", "italian"]
lang_stops = {lang: set(stopwords.words(lang)) for lang in LANGUAGES}

def detect_language(text):
    """Return (best language, share of tokens that are stopwords of that language)."""
    toks = re.findall(r"\w+", text.lower())
    if not toks:
        return "unknown", 0.0
    shares = {lang: sum(t in stops for t in toks) / len(toks) for lang, stops in lang_stops.items()}
    best = max(shares, key=shares.get)
    return best, round(shares[best], 2)

texts = [
    "I really enjoyed the training session today.",
    "Le produit est arrivé en retard et il était cassé.",
    "El servicio al cliente fue excelente y muy rápido.",
    "Das Essen war hervorragend, wir kommen gerne wieder.",
    "Great value for money, would recommend to a friend!",
]

df = pd.DataFrame({"text": texts})
df["n_chars"] = df["text"].str.len()
df["n_tokens"] = df["text"].apply(lambda t: len(word_tokenize(t)))
df["avg_word_len"] = df["text"].apply(lambda t: round(np.mean([len(w) for w in re.findall(r"\w+", t)]), 2))
df["language"] = df["text"].apply(lambda t: detect_language(t)[0])
df["stopword share"] = df["text"].apply(lambda t: detect_language(t)[1])
df

In [ ]:
# 2.6 EXTRA: MORE HAND-MADE FEATURES, AND WHEN LANGUAGE DETECTION FAILS
msgs = pd.DataFrame({"text": [
    "WIN A FREE PRIZE NOW!!! Click http://spam.example to claim",
    "Hi Sam, are we still meeting at 3pm tomorrow?",
    "URGENT!!! Your account is suspended, verify now",
]})
msgs["n_exclaims"] = msgs["text"].str.count("!")
msgs["caps_ratio"] = msgs["text"].apply(
    lambda t: round(sum(c.isupper() for c in t) / max(1, sum(c.isalpha() for c in t)), 2))
msgs["has_url"] = msgs["text"].str.contains(r"https?://", regex=True)
print(msgs[["n_exclaims", "caps_ratio", "has_url"]])

# Language detection on very short and mixed text is unreliable
for t in ["The weather is lovely today in London", "ok", "Bonjour, how are you? Ça va bien."]:
    print(repr(t), "->", detect_language(t))

# Part 3: TF-IDF, Named Entity Recognition and Working with a Dataset

In [ ]:
# 3.1 VOCABULARIES AND DOCUMENT FREQUENCY
corpus_docs = [
    "Inflation rose again this month as energy prices and food prices climbed.",
    "The central bank raised interest rates to fight inflation and protect savings.",
    "The striker scored twice as the home team won the league match.",
    "A late goal gave the visitors a draw in a tense football match.",
    "The new phone has a faster processor and a better camera than last year's model.",
    "Software updates improve the phone battery and fix several security bugs.",
]

tokenised = [preprocess(d) for d in corpus_docs]       # list of token lists, one per document
for i, toks in enumerate(tokenised):
    print(i, toks)

# Vocabulary: every distinct word gets an ID (sorted so the numbering is stable)
all_words = sorted({w for toks in tokenised for w in toks})
vocab = {w: i for i, w in enumerate(all_words)}
print("\nVocabulary size:", len(vocab))
print("First five entries:", dict(list(vocab.items())[:5]))

# Document frequency: in how many documents does each word appear? (set() counts once per document)
doc_freq = FreqDist()
for toks in tokenised:
    doc_freq.update(set(toks))
print("\nWords in the most documents:", doc_freq.most_common(5))
print("Words in only one document :", sum(1 for w, c in doc_freq.items() if c == 1), "of", len(vocab))

# Turn a document into a list of word IDs (unknown words are skipped)
def encode(tokens):
    return [vocab[t] for t in tokens if t in vocab]

print("\nDoc 1 as IDs:", encode(tokenised[1]))
print("New text    :", encode(preprocess("Inflation and football and zebras")))   # 'zebras' is not in the vocabulary

In [ ]:
# 3.1 EXTRA: DECODING AND THE DOCUMENT-TERM TABLE
id_to_word = {i: w for w, i in vocab.items()}
print("Decoded:", [id_to_word[i] for i in encode(tokenised[1])])

dtm_small = build_dtm(tokenised)
print("Table shape:", dtm_small.shape)
dtm_small.loc[:, ["inflation", "price", "match", "phone"]]       # four columns that separate the topics

In [ ]:
# 3.2 TF-IDF THREE WAYS, ALL WITH NLTK
# Uses 'tokenised' from 3.1 (one list of tokens per document).

# --- Way 1: by hand for one term, to show the idea ---
N = len(tokenised)                                    # number of documents
term = "inflation"
df_count = doc_freq[term]                             # document frequency from 3.1
idf = math.log(N / df_count)                          # rarer term = bigger number
tf_doc0 = tokenised[0].count(term)                    # raw count in document 0
print(f"'{term}': in {df_count} of {N} docs, idf = {idf:.3f}, tf in doc0 = {tf_doc0}, tf-idf = {tf_doc0 * idf:.3f}")
print("A word in every document would have idf =", math.log(N / N))     # ln(1) = 0, so it carries no weight

# --- Way 2: NLTK TextCollection ---
from nltk.text import TextCollection

collection = TextCollection(tokenised)                # one token list per document

print("\nNLTK for", repr(term), "in doc 0:")
print("  idf   :", round(collection.idf(term), 3))                      # same as Way 1
print("  tf    :", round(collection.tf(term, tokenised[0]), 3))        # count / document length
print("  tf-idf:", round(collection.tf_idf(term, tokenised[0]), 3))    # tf x idf
print("  unknown word idf:", collection.idf("zebra"))                  # not in the collection: 0, not an error

print("\nNLTK top 3 per document:")
for i, doc_tokens in enumerate(tokenised):
    scores = {t: collection.tf_idf(t, doc_tokens) for t in set(doc_tokens)}
    top3 = sorted(scores.items(), key=lambda kv: -kv[1])[:3]
    print(f"  Doc {i}:", {t: round(sc, 3) for t, sc in top3})

# --- Way 3: the smoothed, length-normalised variant, written with NLTK building blocks ---
def tfidf_vectors(token_docs, smooth=True, normalise=True):
    """Return (vocabulary list, one {word: weight} dictionary per document)."""
    n = len(token_docs)
    df = FreqDist()
    for toks in token_docs:
        df.update(set(toks))
    idf_map = {w: (math.log((1 + n) / (1 + c)) + 1 if smooth else math.log(n / c)) for w, c in df.items()}
    vectors = []
    for toks in token_docs:
        counts = FreqDist(toks)
        vec = {w: counts[w] * idf_map[w] for w in counts}
        if normalise:
            length = math.sqrt(sum(v * v for v in vec.values()))
            vec = {w: v / length for w, v in vec.items()}
        vectors.append(vec)
    return sorted(df), vectors

vocab_list, tfidf_vecs = tfidf_vectors(tokenised)
print("\nSmoothed variant, top 3 per document:")
for i, vec in enumerate(tfidf_vecs):
    top = sorted(vec.items(), key=lambda kv: -kv[1])[:3]
    print(f"  Doc {i}:", {t: round(sc, 2) for t, sc in top})

# The weights as a table (rows documents, columns words), a peek at four columns
tfidf_table = pd.DataFrame([[v.get(w, 0.0) for w in vocab_list] for v in tfidf_vecs],
                           columns=vocab_list).round(2)
tfidf_table[["inflation", "price", "match", "phone"]]

In [ ]:
# 3.2 EXTRA: MINIMAL NLTK TF-IDF EXAMPLE
from nltk.text import TextCollection

docs = [
    "the cat sat on the mat".split(),
    "the dog sat on the log".split(),
    "cats and dogs are pets".split(),
]
tc = TextCollection(docs)

for word in ["the", "cat", "sat", "pets"]:
    print(f"{word:<5} idf = {tc.idf(word):.3f}   tf-idf in doc 0 = {tc.tf_idf(word, docs[0]):.3f}")
# 'the' is in 2 of 3 documents so its idf is low. 'cat' and 'pets' are rarer so they score higher.
# 'pets' is absent from doc 0, so its tf is 0 and its tf-idf there is 0.

In [ ]:
# 3.2 EXTRA: HOW THE CHOICES CHANGE THE RANKING
for smooth, normalise in [(False, False), (True, False), (False, True), (True, True)]:
    _, vecs = tfidf_vectors(tokenised, smooth=smooth, normalise=normalise)
    top = sorted(vecs[1].items(), key=lambda kv: -kv[1])[:4]
    print(f"smooth={smooth!s:<5} normalise={normalise!s:<5}", [(t, round(sc, 2)) for t, sc in top])

# Sublinear tf (1 + ln count) stops a word repeated 20 times from dominating
for count in (1, 2, 5, 20):
    print(f"count {count:>2}:  raw tf = {count:>2}   sublinear tf = {1 + math.log(count):.2f}")

In [ ]:
# 3.3 DOCUMENT SIMILARITY WITH TF-IDF
from nltk.cluster.util import cosine_distance

# Turn each {word: weight} dictionary into a plain list in vocabulary order
matrix = [[vec.get(w, 0.0) for w in vocab_list] for vec in tfidf_vecs]

def similarity(a, b):
    """Cosine similarity (1 = same direction). NLTK returns a distance, so subtract from 1."""
    return 1 - cosine_distance(a, b)

n = len(matrix)
sim = np.array([[similarity(matrix[i], matrix[j]) for j in range(n)] for i in range(n)])

# Most similar partner for every document
for i in range(n):
    others = [(j, sim[i, j]) for j in range(n) if j != i]
    best = max(others, key=lambda x: x[1])
    print(f"Doc {i} is most like doc {best[0]} (similarity {best[1]:.2f})")

# Heatmap of the similarity matrix
plt.figure(figsize=(4.5, 4))
plt.imshow(sim, cmap="Blues", vmin=0, vmax=1)
plt.colorbar(label="cosine similarity")
plt.xticks(range(n)); plt.yticks(range(n))
plt.title("Document similarity")
plt.show()

In [ ]:
# 3.3 EXTRA: A TINY SEARCH ENGINE
def search(query, top_k=3):
    q_tokens = preprocess(query)
    q_vec = {w: 1.0 for w in q_tokens if w in vocab}              # simple query vector: 1 for each known word
    if not q_vec:
        return []
    q_list = [q_vec.get(w, 0.0) for w in vocab_list]
    ranked = sorted(((similarity(q_list, row), i) for i, row in enumerate(matrix)), reverse=True)
    return [(i, round(s, 2)) for s, i in ranked[:top_k] if s > 0]

for q in ["interest rates and inflation", "football match", "phone battery"]:
    print(q, "->", search(q))

In [ ]:
# 3.4 NAMED ENTITY RECOGNITION WITH NLTK
from nltk.tree import Tree

text = ("Barack Obama visited Google in London and met Angela Merkel at the United Nations "
        "on Monday. Apple opened a new office in Berlin.")

tagged = pos_tag(word_tokenize(text))      # step 1 and 2: tokens with part of speech tags
tree = ne_chunk(tagged)                    # step 3: group tagged tokens into entities
print(tree)

def extract_entities(tree):
    """Walk the chunk tree and return a list of (entity text, label)."""
    entities = []
    for node in tree:
        if isinstance(node, Tree):                              # entity subtrees have a label()
            name = " ".join(word for word, tag in node.leaves())
            entities.append((name, node.label()))
    return entities

entities = extract_entities(tree)
pd.DataFrame(entities, columns=["entity", "type"])

In [ ]:
# 3.4 EXTRA: ENTITY COUNTS OVER REAL NEWS
def entities_in(text):
    return extract_entities(ne_chunk(pos_tag(word_tokenize(text))))

sample_ids = [f for f in reuters.fileids() if len(reuters.categories(f)) == 1][:40]
by_type = defaultdict(Counter)
for fid in sample_ids:
    for name, label in entities_in(reuters.raw(fid)[:800]):       # first 800 characters keeps it quick (ne_chunk is slow)
        by_type[label][name] += 1

for label in ["ORGANIZATION", "GPE", "PERSON"]:
    print(label, by_type[label].most_common(6))
# Look for mistakes: stock exchange abbreviations and company names are often mislabelled.
# That is expected, and a good reminder to inspect NER output before relying on it.

# Binary mode: only "is it an entity?", without the type
print(ne_chunk(tagged, binary=True).pformat()[:200])

In [ ]:
# 3.5 LOADING A REAL DATASET: REUTERS TOPICS
topics = ["earn", "acq", "crude"]
per_class = 300

rows = []
for topic in topics:
    ids = [f for f in reuters.fileids(topic) if len(reuters.categories(f)) == 1][:per_class]
    for fid in ids:
        rows.append({"id": fid, "label": topic, "text": reuters.raw(fid).strip()})

news_df = pd.DataFrame(rows).sample(frac=1, random_state=42).reset_index(drop=True)     # shuffle

print("Documents:", len(news_df))
print("\nClass balance:")
print(news_df["label"].value_counts())

news_df["n_words"] = news_df["text"].apply(lambda t: len(word_tokenize(t)))
print("\nWords per document:")
print(news_df["n_words"].describe().round(1))

# Length distribution by class (outliers hidden so a few huge stories do not squash the chart)
news_df.boxplot(column="n_words", by="label", figsize=(7, 4), showfliers=False)
plt.suptitle("")
plt.title("Story length by topic")
plt.ylabel("Words")
plt.show()

# Peek at one story per class
for lab in topics:
    print(f"\n[{lab}]", news_df[news_df.label == lab].text.iloc[0][:200].replace("\n", " "))

In [ ]:
# 3.5 EXTRA: TOP WORDS PER TOPIC (NO MODEL NEEDED)
for lab in topics:
    toks = [w for t in news_df[news_df.label == lab].text for w in preprocess_quick(t)]
    print(f"{lab:<6}", [w for w, _ in FreqDist(toks).most_common(12)])
# earn: 'ct', 'net', 'shr', 'loss' ... acq: 'company', 'share', 'stake' ... crude: 'oil', 'barrel', 'opec'

# Part 4: Supervised Learning, Sentiment and Capstone

In [ ]:
# 4.1 NAIVE BAYES ON REUTERS TOPICS
# Uses news_df from 3.5 (balanced: earn, acq, crude)
from nltk import NaiveBayesClassifier
from nltk.classify import accuracy
from nltk.metrics import precision, recall, f_measure

news_df["tokens"] = news_df["text"].apply(preprocess_quick)           # clean tokens per story

# 1. Split: first 80% to train, last 20% to test (rows were already shuffled in 3.5)
split = int(len(news_df) * 0.8)
train_df, test_df = news_df.iloc[:split], news_df.iloc[split:]
print("Train:", len(train_df), " Test:", len(test_df))

# 2. Features: vocabulary is the 2000 most common words in the TRAINING stories only
train_freq = FreqDist(w for toks in train_df["tokens"] for w in toks)
top_words = [w for w, _ in train_freq.most_common(2000)]
print("Most common words:", top_words[:10])

def doc_features(tokens):
    """One True/False entry for EVERY vocabulary word (NLTK Naive Bayes needs the absences)."""
    present = set(tokens)
    return {w: (w in present) for w in top_words}

train_set = [(doc_features(t), lab) for t, lab in zip(train_df["tokens"], train_df["label"])]
test_set = [(doc_features(t), lab) for t, lab in zip(test_df["tokens"], test_df["label"])]

# 3. Train
t0 = time.time()
nb = NaiveBayesClassifier.train(train_set)
print(f"Trained in {time.time() - t0:.1f}s")

# 4. Evaluate on the test set only
print("Test accuracy:", round(accuracy(nb, test_set), 3))

# Which words does the model rely on most?
nb.show_most_informative_features(10)

# Confusion matrix: rows are the actual class, columns are the prediction
gold = [lab for _, lab in test_set]
pred = [nb.classify(feats) for feats, _ in test_set]
cm = nltk.ConfusionMatrix(gold, pred)
print()
print(cm.pretty_format(sort_by_count=True))

# Precision, recall and F-measure per class, using NLTK's metrics on sets of document numbers
for lab in sorted(set(gold)):
    ref = {i for i, g in enumerate(gold) if g == lab}          # documents that truly are this class
    tst = {i for i, p in enumerate(pred) if p == lab}          # documents the model called this class
    print(f"{lab:<6} precision {precision(ref, tst):.2f}  recall {recall(ref, tst):.2f}  F {f_measure(ref, tst):.2f}")

In [ ]:
# 4.1 EXTRA: CONFUSION HEATMAP AND PROBABILITIES
labels = sorted(set(gold))
grid = pd.crosstab(pd.Series(gold, name="actual"), pd.Series(pred, name="predicted")).reindex(
    index=labels, columns=labels, fill_value=0)

plt.figure(figsize=(4.8, 4))
plt.imshow(grid.values, cmap="Blues")
for i in range(len(labels)):
    for j in range(len(labels)):
        plt.text(j, i, grid.values[i, j], ha="center", va="center")
plt.xticks(range(len(labels)), labels); plt.yticks(range(len(labels)), labels)
plt.xlabel("predicted"); plt.ylabel("actual"); plt.title("Confusion matrix")
plt.show()

# Probabilities for one test story
feats, true_label = test_set[0]
dist = nb.prob_classify(feats)
print("True label:", true_label)
print({lab: round(dist.prob(lab), 3) for lab in dist.samples()})
print("Predicted :", dist.max())

In [ ]:
# 4.1 EXTRA: WHY SMOOTHING MATTERS, BY HAND
# Suppose 100 'earn' stories contain 'oil' in 0 of them, and 100 'crude' stories contain 'oil' in 40.
earn_with_oil, crude_with_oil, n_each = 0, 40, 100

print("Without smoothing: P(oil | earn) =", earn_with_oil / n_each)           # exactly 0, vetoes 'earn' for ANY story with 'oil'
smoothed = (earn_with_oil + 1) / (n_each + 2)                                 # add one to each side of the True/False split
print("With smoothing   : P(oil | earn) =", round(smoothed, 4))              # small but not zero

# NLTK lets you choose the smoothing method when you train
from nltk.probability import LaplaceProbDist
nb_laplace = NaiveBayesClassifier.train(train_set, estimator=LaplaceProbDist)
print("\nDefault smoothing accuracy :", round(accuracy(nb, test_set), 3))
print("Laplace smoothing accuracy :", round(accuracy(nb_laplace, test_set), 3))

In [ ]:
# 4.2 FEATURE CHOICES AND TUNING
def make_features(vocab_words, tokens, include_absent=True):
    present = set(tokens)
    if include_absent:
        return {w: (w in present) for w in vocab_words}
    return {w: True for w in vocab_words if w in present}          # only the words that appear

def run(vocab_size, include_absent=True):
    vocab_words = [w for w, _ in train_freq.most_common(vocab_size)]
    tr = [(make_features(vocab_words, t, include_absent), lab) for t, lab in zip(train_df["tokens"], train_df["label"])]
    te = [(make_features(vocab_words, t, include_absent), lab) for t, lab in zip(test_df["tokens"], test_df["label"])]
    return accuracy(NaiveBayesClassifier.train(tr), te)

rows = []
for size in (100, 500, 2000):
    rows.append((size, round(run(size, True), 3), round(run(size, False), 3)))
pd.DataFrame(rows, columns=["vocabulary size", "accuracy (with False entries)", "accuracy (present words only)"])

In [ ]:
# 4.3 LOGISTIC REGRESSION THROUGH NLTK'S SklearnClassifier
from nltk.classify import SklearnClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfTransformer

# 1. Load the 2000 reviews and shuffle them
reviews = [(movie_reviews.raw(fid), movie_reviews.categories(fid)[0]) for fid in movie_reviews.fileids()]
random.Random(42).shuffle(reviews)

def to_tokens(text):
    """Lowercase word tokens, letters only. Stopwords are KEPT for sentiment."""
    return [w for w in word_tokenize(text.lower()) if w.isalpha()]

review_tokens = [to_tokens(text) for text, _ in reviews]
review_labels = [lab for _, lab in reviews]

# 2. Split 80 / 20 and choose the 3000 most common training words as the vocabulary
cut = int(len(reviews) * 0.8)
rev_freq = FreqDist(w for toks in review_tokens[:cut] for w in toks)
rev_vocab = [w for w, _ in rev_freq.most_common(3000)]
vocab_set = set(rev_vocab)

def count_features(tokens):
    """Word counts for vocabulary words only (a sparse dictionary)."""
    return dict(Counter(t for t in tokens if t in vocab_set))

rev_train = [(count_features(t), lab) for t, lab in zip(review_tokens[:cut], review_labels[:cut])]
rev_test = [(count_features(t), lab) for t, lab in zip(review_tokens[cut:], review_labels[cut:])]
print("Train:", len(rev_train), " Test:", len(rev_test), " Vocabulary:", len(rev_vocab))

# 3. Logistic regression with TF-IDF weights, trained and used through NLTK
def make_logreg(C=10):
    return SklearnClassifier(Pipeline([("tfidf", TfidfTransformer()),
                                       ("clf", LogisticRegression(C=C, max_iter=1000))]))

t0 = time.time()
logreg = make_logreg(C=10).train(rev_train)
print(f"Logistic regression: accuracy {accuracy(logreg, rev_test):.3f}  (fit {time.time() - t0:.1f}s)")

# Compare with Naive Bayes on True/False features
def nb_feats(toks):
    present = set(toks)                                   # build the set once per review, not once per word
    return {w: (w in present) for w in rev_vocab[:2000]}
t0 = time.time()
nb_reviews = NaiveBayesClassifier.train([(nb_feats(t), lab) for t, lab in zip(review_tokens[:cut], review_labels[:cut])])
nb_acc = accuracy(nb_reviews, [(nb_feats(t), lab) for t, lab in zip(review_tokens[cut:], review_labels[cut:])])
print(f"Naive Bayes        : accuracy {nb_acc:.3f}  (fit {time.time() - t0:.1f}s)")

# 4. Effect of C: training vs test accuracy
Cs = [0.01, 0.1, 1, 10, 100, 1000]
train_acc, test_acc = [], []
for C in Cs:
    m = make_logreg(C).train(rev_train)
    train_acc.append(accuracy(m, rev_train))
    test_acc.append(accuracy(m, rev_test))

plt.figure(figsize=(6.5, 3.8))
plt.semilogx(Cs, train_acc, marker="o", label="train")
plt.semilogx(Cs, test_acc, marker="o", label="test")
plt.xlabel("C (higher = weaker regularisation)")
plt.ylabel("Accuracy")
plt.title("Logistic regression: train vs test")
plt.legend()
plt.show()

In [ ]:
# 4.3 EXTRA: NEGATION MARKING
from nltk.sentiment.util import mark_negation

print(mark_negation("this film was not good at all , but the acting was fine".split()))

# Build a new vocabulary and features from the negation-marked tokens (training data only)
# Mark negation on the raw tokens, before punctuation is dropped, so each negation stops at the next comma or full stop
neg_tokens = []
for text, _ in reviews:
    marked = mark_negation(word_tokenize(text.lower()))
    neg_tokens.append([w for w in marked if w.replace("_NEG", "").isalpha()])
neg_freq = FreqDist(w for toks in neg_tokens[:cut] for w in toks)
neg_vocab = set(w for w, _ in neg_freq.most_common(3000))

def neg_features(tokens):
    return dict(Counter(t for t in tokens if t in neg_vocab))

neg_train = [(neg_features(t), lab) for t, lab in zip(neg_tokens[:cut], review_labels[:cut])]
neg_test = [(neg_features(t), lab) for t, lab in zip(neg_tokens[cut:], review_labels[cut:])]
logreg_neg = make_logreg(C=10).train(neg_train)
print("Without negation marking:", round(accuracy(logreg, rev_test), 3))
print("With negation marking   :", round(accuracy(logreg_neg, neg_test), 3))
# On long reviews the change is small and can go either way. The idea matters more for short messages.

In [ ]:
# 4.3 EXTRA: UNIGRAMS PLUS BIGRAMS
def with_bigrams(tokens):
    return tokens + [a + "_" + b for a, b in ngrams(tokens, 2)]

bi_tokens = [with_bigrams(t) for t in review_tokens]
bi_freq = FreqDist(w for toks in bi_tokens[:cut] for w in toks)
bi_vocab = set(w for w, c in bi_freq.most_common(8000) if c >= 3)       # keep pairs seen at least 3 times

def bi_features(tokens):
    return dict(Counter(t for t in tokens if t in bi_vocab))

bi_train = [(bi_features(t), lab) for t, lab in zip(bi_tokens[:cut], review_labels[:cut])]
bi_test = [(bi_features(t), lab) for t, lab in zip(bi_tokens[cut:], review_labels[cut:])]
logreg_bi = make_logreg(C=10).train(bi_train)
print("Unigrams only      :", round(accuracy(logreg, rev_test), 3), " features:", len(vocab_set))
print("Unigrams + bigrams :", round(accuracy(logreg_bi, bi_test), 3), " features:", len(bi_vocab))

In [ ]:
# 4.3b SUPPORT VECTOR MACHINES THROUGH NLTK (reuses rev_train and rev_test from 4.3)
from sklearn.svm import LinearSVC

def make_svm(C=1.0):
    return SklearnClassifier(Pipeline([("tfidf", TfidfTransformer()),
                                       ("clf", LinearSVC(C=C, max_iter=5000))]))

svm = make_svm(C=1.0).train(rev_train)
print("LinearSVC accuracy:", round(accuracy(svm, rev_test), 3))

# Precision and recall per class, from NLTK's metrics
svm_gold = [lab for _, lab in rev_test]
svm_pred = [svm.classify(f) for f, _ in rev_test]
print(nltk.ConfusionMatrix(svm_gold, svm_pred).pretty_format(sort_by_count=True))
for lab in ("neg", "pos"):
    ref = {i for i, g in enumerate(svm_gold) if g == lab}
    tst = {i for i, p in enumerate(svm_pred) if p == lab}
    print(f"{lab}: precision {precision(ref, tst):.2f}  recall {recall(ref, tst):.2f}  F {f_measure(ref, tst):.2f}")

# Effect of C on train vs test accuracy
Cs_svm = [0.001, 0.01, 0.1, 1, 10, 100]
svm_train, svm_test = [], []
for C in Cs_svm:
    m = make_svm(C).train(rev_train)
    svm_train.append(accuracy(m, rev_train))
    svm_test.append(accuracy(m, rev_test))

plt.figure(figsize=(6.5, 3.8))
plt.semilogx(Cs_svm, svm_train, marker="o", label="train")
plt.semilogx(Cs_svm, svm_test, marker="o", label="test")
plt.xlabel("C (higher = tighter fit to training data)")
plt.ylabel("Accuracy")
plt.title("LinearSVC: train vs test")
plt.legend()
plt.show()

# Head to head on identical features
rows = []
for name, build in [("Logistic regression", lambda: make_logreg(10)), ("Linear SVM", lambda: make_svm(1.0))]:
    t0 = time.time()
    model = build().train(rev_train)
    rows.append((name, round(accuracy(model, rev_test), 3), round(time.time() - t0, 2)))
rows.append(("Naive Bayes (True/False features)", round(nb_acc, 3), "n/a"))
print()
print(pd.DataFrame(rows, columns=["model", "test accuracy", "fit time (s)"]))

# Interpret: the words with the strongest weights. NLTK keeps the fitted scikit-learn parts in _clf and _vectorizer.
fitted = svm._clf.named_steps["clf"]
names = svm._vectorizer.get_feature_names_out()
weights = pd.Series(fitted.coef_[0], index=names)
positive_class = fitted.classes_[1]                      # positive weights point towards this class
print("\nPositive weights point towards:", repr(positive_class))
print("Strongest towards 'pos':", list(weights.sort_values(ascending=(positive_class != "pos")).head(10).index))
print("Strongest towards 'neg':", list(weights.sort_values(ascending=(positive_class == "pos")).head(10).index))

In [ ]:
# 4.3b EXTRA: RBF KERNEL VERSUS LINEAR
from sklearn.svm import SVC

t0 = time.time()
rbf = SklearnClassifier(Pipeline([("tfidf", TfidfTransformer()), ("clf", SVC(kernel="rbf", C=10))])).train(rev_train)
rbf_time = time.time() - t0
t0 = time.time()
lin = make_svm(1.0).train(rev_train)
lin_time = time.time() - t0
print(f"RBF SVM    : accuracy {accuracy(rbf, rev_test):.3f}   fit {rbf_time:.1f}s")
print(f"Linear SVM : accuracy {accuracy(lin, rev_test):.3f}   fit {lin_time:.1f}s")

In [ ]:
# 4.4 SENTIMENT WITH VADER
from nltk.sentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()
for text in ["I love this phone, the camera is fantastic!",
             "This is not good. Really disappointing.",
             "The delivery arrived on Tuesday.",
             "Absolutely TERRIBLE service!!!"]:
    print(f"{sia.polarity_scores(text)['compound']:+.2f}  {text}")

# How good is VADER on the same 400 test reviews the SVM saw?
test_texts = [text for text, _ in reviews[cut:]]
vader_pred = ["pos" if sia.polarity_scores(t)["compound"] >= 0 else "neg" for t in test_texts]
vader_acc = sum(p == g for p, g in zip(vader_pred, review_labels[cut:])) / len(vader_pred)
print(f"\nVADER accuracy on long reviews: {vader_acc:.3f}")
print(f"Linear SVM accuracy           : {accuracy(svm, rev_test):.3f}")

In [ ]:
# 4.4 EXTRA: UPDATE THE LEXICON
before = sia.polarity_scores("The new update is a total banger, absolutely sick!")["compound"]
sia.lexicon.update({"banger": 3.0, "sick": 2.0})          # scores run from about -4 to +4
after = sia.polarity_scores("The new update is a total banger, absolutely sick!")["compound"]
print(f"Before: {before:+.2f}   After: {after:+.2f}")

In [ ]:
# 4.5 READING THE MODEL: INFORMATIVE WORDS AND MISTAKES
print("Naive Bayes, most informative words on reviews:")
nb_reviews.show_most_informative_features(8)

# The logistic regression's most confident mistakes
errors = []
for (text, gold_label), (feats, _) in zip(reviews[cut:], rev_test):
    dist = logreg.prob_classify(feats)
    if dist.max() != gold_label:
        errors.append((dist.prob(dist.max()), gold_label, dist.max(), text))
errors.sort(reverse=True)                                        # most confident wrong answers first

print(f"\nLogistic regression got {len(errors)} of {len(rev_test)} wrong. Three most confident mistakes:")
for conf, g, p, text in errors[:3]:
    print(f"\n  actual {g}, predicted {p} ({conf:.0%} sure)")
    print("  ", textwrap.shorten(text.replace("\n", " "), 220))

In [ ]:
# 4.6 CAPSTONE: A REUSABLE SENTIMENT PIPELINE IN NLTK
from nltk.sentiment.util import mark_negation

def text_to_tokens(text):
    """Raw text in, negation-marked lowercase tokens out."""
    marked = mark_negation(word_tokenize(text.lower()))              # mark first, while punctuation still ends each negation
    return [w for w in marked if w.replace("_NEG", "").isalpha()]

class SentimentModel:
    def __init__(self, C=10, review_band=(0.40, 0.60)):
        self.C, self.band = C, review_band

    def fit(self, texts, labels):
        token_lists = [text_to_tokens(t) for t in texts]
        freq = FreqDist(w for toks in token_lists for w in toks)
        self.vocab = set(w for w, _ in freq.most_common(3000))        # vocabulary from training data only
        data = [(self.features(toks), lab) for toks, lab in zip(token_lists, labels)]
        self.clf = SklearnClassifier(Pipeline([("tfidf", TfidfTransformer()),
                                               ("clf", LogisticRegression(C=self.C, max_iter=1000))])).train(data)
        return self

    def features(self, tokens):
        return dict(Counter(t for t in tokens if t in self.vocab))

    def predict(self, text):
        """Return (label, probability of 'pos'). Label is 'unsure' inside the review band."""
        dist = self.clf.prob_classify(self.features(text_to_tokens(text)))
        p_pos = dist.prob("pos")
        label = "unsure" if self.band[0] <= p_pos <= self.band[1] else ("pos" if p_pos > 0.5 else "neg")
        return label, round(float(p_pos), 2)

# Train on the training reviews, hold out the rest
texts = [t for t, _ in reviews]
labels = [l for _, l in reviews]
model = SentimentModel().fit(texts[:cut], labels[:cut])

# Evaluate honestly on the held-out reviews
results = [model.predict(t) for t in texts[cut:]]
decided = [(p, g) for (p, _), g in zip(results, labels[cut:]) if p != "unsure"]
print(f"Sent to review band: {len(results) - len(decided)} of {len(results)}")
print(f"Accuracy on the rest: {sum(p == g for p, g in decided) / len(decided):.3f}")
print(nltk.ConfusionMatrix([g for _, g in decided], [p for p, _ in decided]).pretty_format(sort_by_count=True))

# Save and reload
with open("sentiment_model.pkl", "wb") as f:
    pickle.dump(model, f)
with open("sentiment_model.pkl", "rb") as f:
    reloaded = pickle.load(f)

# Try it on messages it has never seen
for msg in ["A brilliant, moving and beautifully acted film. I loved it.",
            "Not good. The plot was dull and the acting was worse.",
            "It was fine, I suppose.",
            "The delivery arrived on Tuesday."]:
    label, p_pos = reloaded.predict(msg)
    print(f"{label:<7} P(pos) = {p_pos:.2f} | {msg}")

In [ ]:
# 4.6 EXTRA: DOES THE REVIEW BAND HELP?
probs = [model.clf.prob_classify(model.features(text_to_tokens(t))).prob("pos") for t in texts[cut:]]
truth = labels[cut:]

print(f"{'band':<14}{'to review':>10}{'accuracy on rest':>20}")
for low, high in [(0.5, 0.5), (0.45, 0.55), (0.40, 0.60), (0.30, 0.70)]:
    kept = [(p, g) for p, g in zip(probs, truth) if not (low <= p <= high)]
    acc = sum(("pos" if p > 0.5 else "neg") == g for p, g in kept) / len(kept)
    print(f"{low:.2f}-{high:.2f}     {len(probs) - len(kept):>10}{acc:>20.3f}")
# A wider band sends more to people and the rest becomes more accurate. You choose the balance.

In [ ]:
# 4.7 EXERCISES: space to experiment
# Everything defined earlier is available here: preprocess, preprocess_quick, news_df, train_df, test_df,
# make_logreg, make_svm, rev_train, rev_test, sia, model.

# Exercise 1 starter
for size in (2000, 5000):
    t0 = time.time()
    acc = run(size)
    print(size, round(acc, 3), f"{time.time() - t0:.1f}s")